# CS781 Phase 4 — Experiment 1
## Model4 Partial/Layer-wise FT + Head/Middle/Tail Hierarchical Attention

Built around the Phase-4 audit:
- 32,305 unique TRAIN cases after removing exact duplicate IDs.
- clean DEV = 994 cases (497/497).
- all 1,517 TEST IDs overlap with records in the raw public DEV file, so those overlapping DEV rows are excluded from all training, threshold tuning, checkpoint selection and refit.
- TEST `label` / `expert_*` fields are ignored.
- long documents are represented with HEAD / MIDDLE / TAIL BPE chunks.

Selection uses the exact supplied Phase-4 evaluator and tunes the binary threshold on clean DEV Macro-F1.


### v2 changes retained
- **Faster tokenisation (the biggest time sink):** only the head / middle / tail text *slices* are BPE-encoded instead of whole cases (up to 412k words). Head and tail tokens are identical to full-document tokenisation (slices start at whitespace boundaries). The refit reuses the already-tokenised datasets instead of tokenising train+dev again.
- **Faster training:** padding mask removed (right-padded + causal ⇒ valid positions never see pads), so attention uses the fast causal kernel with identical outputs for valid tokens.
- **Train/test text-style alignment:** train looked mixed-case with no line breaks, dev/test lowercase with hard `\n` breaks. Diagnostics are printed and, if TEST is lowercase, all splits are lowercased and whitespace-collapsed (`CASE_MODE`, `COLLAPSE_WS`).
- **Robust dedupe:** same-ID rows with different text no longer crash the run (keep the longer text, drop label-conflict IDs).
- **Smoothed threshold:** picks the centre of the plateau of the dev F1 curve (moving average) instead of a single lucky point on 994 documents; F1@0.5 is logged too.
- **Refit replays the validated LR schedule** (`REFIT_MATCH_ORIGINAL_SCHEDULE`), as in Phase 3.
- **Label-free calibration of the refit model:** its threshold is set so it predicts the same positive rate on the test inputs as the validated held-out model (no test labels used); the ensemble averages margins relative to each model's own threshold. Variants: heldout / refit / refit_rawthr / ensemble.
- **Report analyses built in (cell 10b):** test-time section ablation (head/middle/tail only, leave-one-out), false-positive/false-negative statistics by length, chunk attention.
- **KD ready:** last cell saves logits on train / clean dev / test (`teacher_logits_cjpe.npz`).

### v3 score-oriented additions
- **4th evidence/disposition chunk:** adds a label-agnostic chunk centred near the last legal disposition cue
  (`appeal allowed/dismissed`, `petition allowed/dismissed`, `set aside`, `acquitted`, etc.); if no cue is found,
  it falls back to a late-document (~75%) chunk. This directly uses the Phase-4 audit finding that outcome evidence
  is concentrated near the end without mapping any cue to a class label.
- **Top-3 checkpoint retention:** keeps the three strongest held-out checkpoints and creates a snapshot ensemble,
  plus a snapshot+refit ensemble.
- **Safer threshold plateau selection:** edge-safe smoothing of the DEV Macro-F1 curve; logs both the selected
  operating-point F1 and the absolute best grid F1.
- **Fast-kernel correctness check:** verifies on a real padded batch that dropping the padding mask changes valid
  token hidden states only within numerical tolerance.
- **Stronger KD exports:** saves held-out teacher logits on TRAIN/CLEAN-DEV and, after refit, refit-teacher logits
  on TRAIN+CLEAN-DEV separately for a future Phase-4 KD experiment.


In [1]:
# CELL 1 — imports/config
import os,re,gc,json,math,time,random,zipfile,subprocess,sys,hashlib,shutil
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import f1_score,precision_score,recall_score
from huggingface_hub import hf_hub_download,login

SEED=1234
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",DEVICE)
if DEVICE.type=="cuda":
    print("GPU:",torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32=True
    torch.backends.cudnn.allow_tf32=True
torch.set_float32_matmul_precision("high")

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN=None
assert HF_TOKEN,"HF_TOKEN missing in Kaggle Secrets"
login(token=HF_TOKEN)

HF_REPO="Exploration-Lab/CS-781-Capstone"
ROOT=Path("/kaggle/working/phase4_exp1_model4_partialft_hier")
ROOT.mkdir(parents=True,exist_ok=True)

VOCAB_SIZE=24576
MAX_POSITIONS=2560
D_MODEL=640
N_LAYERS=17
N_HEADS=10
N_KV_HEADS=2
FFN_DIM=2048
ROPE_BASE=10000.0
RMS_EPS=1e-5
DROPOUT=0.0
EXPECTED_TOKENIZER_SHA256="9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d"

N_CHUNKS=4
CHUNK_LEN=384
CHUNK_NAMES=["head","middle","evidence","tail"]

TRAIN_TOP_N_LAYERS=5
BACKBONE_LR=3e-5
HEAD_LR=5e-4
LAYERWISE_LR_DECAY=0.90
WEIGHT_DECAY=0.01

NUM_EPOCHS=2
EVAL_PER_EPOCH=3
PATIENCE_EVALS=4
MICRO_BATCH_SIZE=2
GRAD_ACCUM_STEPS=8
EVAL_BATCH_SIZE=6
WARMUP_RATIO=0.06
GRAD_CLIP=1.0
LABEL_SMOOTHING=0.02
USE_AMP=True
AMP_INIT_SCALE=4096.0
NUM_WORKERS=2

CHUNK_PROJ_DIM=384
HEAD_DROPOUT=0.15
THRESHOLDS=np.round(np.arange(0.20,0.801,0.005),3)

RUN_REFIT=True
ENSEMBLE_HELDOUT_REFIT=True
REFIT_MATCH_ORIGINAL_SCHEDULE=True   # replay the LR schedule the validated run used
THRESH_SMOOTH=9                      # moving-average width (in 0.005 steps) for threshold choice
CASE_MODE="auto"                     # "auto" | "lower" | "keep"; auto lowercases if TEST is lowercase
COLLAPSE_WS=True                     # collapse \n/runs of spaces to one space (test has hard line breaks)
EXPORT_TEACHER_LOGITS=True
TOPK_SNAPSHOTS=3
MAKE_SNAPSHOT_ENSEMBLES=True           # last cell: logits on train/dev/test for the KD experiment

print("chunks:",N_CHUNKS,"x",CHUNK_LEN)
print("effective document batch:",MICRO_BATCH_SIZE*GRAD_ACCUM_STEPS)


# Label-agnostic evidence selector. These phrases only choose a region;
# they are never mapped to class 0/1.
EVIDENCE_CUES=[
    "appeal is allowed","appeal allowed",
    "appeal is dismissed","appeal dismissed",
    "petition is allowed","petition allowed",
    "petition is dismissed","petition dismissed",
    "we allow","we dismiss",
    "is hereby allowed","is hereby dismissed",
    "set aside","acquitted","convicted",
    "in the result","accordingly",
]


Device: cuda
GPU: Tesla T4
chunks: 4 x 384
effective document batch: 16


In [2]:
# CELL 2 — locate checkpoint/tokenizer
def all_files():
    for base in ["/kaggle/input","/kaggle/working"]:
        if not os.path.isdir(base): continue
        for root,_,files in os.walk(base):
            for fn in files:
                yield os.path.join(root,fn)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""):
            h.update(b)
    return h.hexdigest()

files=list(all_files())
ckpts=[p for p in files if p.lower().endswith(".pt") and "eff_full_best" in os.path.basename(p).lower()]
toks=[p for p in files if os.path.basename(p).lower()=="tokenizer.json"]

assert ckpts,"Attach Phase-1 Model4 eff_full_best*.pt"
assert toks,"Attach tokenizer.json"

PHASE1_CKPT=sorted(ckpts,key=lambda p:(sum(w in p.lower() for w in ["model4","phase1","efficient","eff_full_best"]),p),reverse=True)[0]
exact=[p for p in toks if sha256_file(p)==EXPECTED_TOKENIZER_SHA256]
assert exact,"Exact Phase-1 tokenizer not found"
TOKENIZER_FILE=exact[0]

print("checkpoint:",PHASE1_CKPT)
print("tokenizer:",TOKENIZER_FILE)
print("SHA:",sha256_file(TOKENIZER_FILE))


checkpoint: /kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt
tokenizer: /kaggle/input/datasets/aaryant23/phase2-model4/tokenizer.json
SHA: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [3]:
# CELL 3 — exact LegalBPE loader
BPE_SPECIALS=["<pad>","<unk>","<s>","</s>"]
class LegalBPE:
    def __init__(self,vocab_size=5000):
        self.vocab_size=int(vocab_size); self.merges=[]
        self.itos=list(BPE_SPECIALS); self.stoi={t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes={}
        for b in range(256):
            s=f"{b:02x}"; self.symbol_bytes[s]=bytes([b]); self.stoi[s]=len(self.itos); self.itos.append(s)
        self._reset_encode_cache()
    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)
    @staticmethod
    def word_to_symbols(word): return [f"{b:02x}" for b in word.encode("utf-8")]
    def _reset_encode_cache(self):
        self._ranks={(l,r):i for i,(l,r) in enumerate(self.merges)}
        self._word_cache={}; self._cache_nmerges=len(self.merges)
    def encode_word(self,word):
        if self._cache_nmerges!=len(self.merges): self._reset_encode_cache()
        hit=self._word_cache.get(word)
        if hit is not None: return list(hit)
        symbols=self.word_to_symbols(word)
        while len(symbols)>1:
            best=None; rank=None
            for j in range(len(symbols)-1):
                rr=self._ranks.get((symbols[j],symbols[j+1]))
                if rr is not None and (rank is None or rr<rank):
                    best=(symbols[j],symbols[j+1]); rank=rr
            if best is None: break
            l,r=best; merged=[]; j=0
            while j<len(symbols):
                if j+1<len(symbols) and symbols[j]==l and symbols[j+1]==r:
                    merged.append(l+r); j+=2
                else:
                    merged.append(symbols[j]); j+=1
            symbols=merged
        if len(self._word_cache)>=250000: self._word_cache.clear()
        self._word_cache[word]=tuple(symbols)
        return symbols
    def encode(self,text):
        out=[]; unk=self.unk_token_id
        for unit in re.findall(r"\S+|\s+",str(text)):
            for s in self.encode_word(unit): out.append(self.stoi.get(s,unk))
        return out
    def decode(self,ids):
        buf=bytearray()
        for idx in ids:
            tok=self.itos[int(idx)]
            if tok not in BPE_SPECIALS: buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8",errors="replace")
    @classmethod
    def load_json(cls,path):
        obj=json.load(open(path,encoding="utf-8"))
        tok=cls(obj["vocab_size"])
        tok.merges=[list(x) for x in obj["merges"]]
        tok.itos=list(obj["itos"]); tok.stoi={t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size=len(tok.itos); tok.symbol_bytes={}
        for b in range(256):
            s=f"{b:02x}"; tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges: tok.symbol_bytes[l+r]=tok.symbol_bytes[l]+tok.symbol_bytes[r]
        tok._reset_encode_cache(); return tok

tokenizer=LegalBPE.load_json(TOKENIZER_FILE)
assert len(tokenizer)==VOCAB_SIZE
probe="IN THE SUPREME COURT OF INDIA\nSection 14: ₹100 — legal test."
assert tokenizer.decode(tokenizer.encode(probe))==probe
print("Tokenizer PASS:",len(tokenizer))


Tokenizer PASS: 24576


In [4]:
# CELL 4 — download Phase4 + build clean splits (+ train/test text normalisation)
FNS={
 "train":"Phase_4_CJPE/Train/train.json",
 "dev":"Phase_4_CJPE/Train/dev.json",
 "test":"Phase_4_CJPE/Test/test.json",
}
paths={k:hf_hub_download(repo_id=HF_REPO,repo_type="dataset",filename=v,token=HF_TOKEN) for k,v in FNS.items()}
train_raw=json.load(open(paths["train"],encoding="utf-8"))
dev_raw=json.load(open(paths["dev"],encoding="utf-8"))
test_raw=json.load(open(paths["test"],encoding="utf-8"))

def dedup_labeled(rows,name):
    seen={}; conflicts=set(); text_diff=0
    for r in rows:
        cid=str(r["id"]); y=int(r["label"]); txt=str(r["text"])
        assert y in (0,1)
        if cid in seen:
            if seen[cid]["label"]!=y: conflicts.add(cid)
            elif seen[cid]["text"]!=txt:
                text_diff+=1
                if len(txt)>len(seen[cid]["text"]): seen[cid]["text"]=txt   # keep the more complete version
        else:
            seen[cid]={"id":cid,"text":txt,"label":y}
    for c in conflicts: seen.pop(c,None)
    print(f"{name}: rows={len(rows)} unique={len(seen)+len(conflicts)} | same-ID different-text={text_diff} | label-conflict IDs dropped={len(conflicts)}")
    return list(seen.values())

train_records=dedup_labeled(train_raw,"train")
dev_unique=dedup_labeled(dev_raw,"dev")

# TEST: deliberately copy only id/text. Never read/use test label.
test_records=[{"id":str(r["id"]),"text":str(r["text"])} for r in test_raw]
test_ids={r["id"] for r in test_records}
assert len(test_ids)==len(test_records)

clean_dev=[r for r in dev_unique if r["id"] not in test_ids]
overlap_count=sum(r["id"] in test_ids for r in dev_unique)

if len(train_records)!=32305: print("⚠️ train unique count differs from audit:",len(train_records))
assert len(clean_dev)==994
assert len(test_records)==1517
assert overlap_count==1517
assert not ({r["id"] for r in train_records}&test_ids)
assert not ({r["id"] for r in clean_dev}&test_ids)

# ---- text style diagnostics + normalisation (train looked mixed-case, dev/test lowercase) ----
def upper_ratio(t):
    t=t[:20000]; letters=sum(c.isalpha() for c in t); up=sum(c.isupper() for c in t)
    return up/max(letters,1)
_rng=random.Random(SEED)
def sample_ratio(rows,k=1500):
    idx=_rng.sample(range(len(rows)),min(k,len(rows)))
    return np.array([upper_ratio(rows[i]["text"]) for i in idx])
ur_tr,ur_dv,ur_te=sample_ratio(train_records),sample_ratio(clean_dev),sample_ratio(test_records)
print(f"uppercase-letter ratio  train median={np.median(ur_tr):.4f} (>2% in {100*np.mean(ur_tr>0.02):.1f}% of docs)"
      f" | clean-dev median={np.median(ur_dv):.4f} | test median={np.median(ur_te):.4f}")
print("newline-containing docs  train: {:.1f}% | dev: {:.1f}% | test: {:.1f}%".format(
    *[100*np.mean(["\n" in r["text"] for r in rows[:2000]]) for rows in (train_records,clean_dev,test_records)]))

LOWERCASE=(CASE_MODE=="lower") or (CASE_MODE=="auto" and float(np.median(ur_te))<0.01)
print("LOWERCASE:",LOWERCASE,"| COLLAPSE_WS:",COLLAPSE_WS)
def normalize_text(t):
    if LOWERCASE: t=t.lower()
    if COLLAPSE_WS: t=re.sub(r"\s+"," ",t).strip()
    return t
for rows in (train_records,clean_dev,test_records):
    for r in rows: r["text"]=normalize_text(r["text"])
del train_raw,dev_raw,test_raw,dev_unique; gc.collect()

print("TRAIN:",len(train_records),Counter(r["label"] for r in train_records))
print("CLEAN DEV:",len(clean_dev),Counter(r["label"] for r in clean_dev))
print("TEST:",len(test_records))
print("Excluded DEV↔TEST overlap:",overlap_count)
print("✅ TEST label/expert fields ignored.")


Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 80.3MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

train: rows=37387 unique=32305 | same-ID different-text=0 | label-conflict IDs dropped=0
dev: rows=3505 unique=2511 | same-ID different-text=0 | label-conflict IDs dropped=0
uppercase-letter ratio  train median=0.0351 (>2% in 97.1% of docs) | clean-dev median=0.0000 | test median=0.0000
newline-containing docs  train: 0.0% | dev: 100.0% | test: 98.9%
LOWERCASE: True | COLLAPSE_WS: True
TRAIN: 32305 Counter({0: 18920, 1: 13385})
CLEAN DEV: 994 Counter({0: 497, 1: 497})
TEST: 1517
Excluded DEV↔TEST overlap: 1517
✅ TEST label/expert fields ignored.


In [5]:
# CELL 5 — exact Model4 architecture
class RMSNorm(nn.Module):
    def __init__(self,dim,eps=1e-5):
        super().__init__(); self.weight=nn.Parameter(torch.ones(dim)); self.eps=eps
    def forward(self,x):
        dtype=x.dtype; x32=x.float()
        x32=x32*torch.rsqrt(x32.pow(2).mean(-1,keepdim=True)+self.eps)
        return x32.to(dtype)*self.weight

class RotaryTable(nn.Module):
    def __init__(self,head_dim,max_positions,base=10000.0):
        super().__init__()
        inv=1.0/(base**(torch.arange(0,head_dim,2,dtype=torch.float32)/head_dim))
        freqs=torch.outer(torch.arange(max_positions,dtype=torch.float32),inv)
        self.register_buffer("cos_cached",freqs.cos(),persistent=False)
        self.register_buffer("sin_cached",freqs.sin(),persistent=False)
    def forward(self,position_ids):
        return self.cos_cached[position_ids].unsqueeze(1),self.sin_cached[position_ids].unsqueeze(1)

def apply_rope(x,cos,sin):
    B,H,T,D=x.shape; cos=cos.to(x.dtype); sin=sin.to(x.dtype)
    xp=x.view(B,H,T,D//2,2); x0,x1=xp[...,0],xp[...,1]
    return torch.stack((x0*cos-x1*sin,x0*sin+x1*cos),dim=-1).flatten(-2)

class GQACausalSelfAttention(nn.Module):
    def __init__(self,d_model,n_heads,n_kv_heads,dropout):
        super().__init__(); self.n_heads=n_heads; self.n_kv_heads=n_kv_heads
        self.head_dim=d_model//n_heads; self.kv_repeat=n_heads//n_kv_heads; self.dropout=dropout
        self.q_proj=nn.Linear(d_model,n_heads*self.head_dim,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.head_dim,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.head_dim,bias=False)
        self.o_proj=nn.Linear(d_model,d_model,bias=False)
    def forward(self,x,cos,sin,key_valid=None):
        B,T,C=x.shape; H=self.n_heads; KV=self.n_kv_heads; D=self.head_dim
        q=self.q_proj(x).view(B,T,H,D).transpose(1,2)
        k=self.k_proj(x).view(B,T,KV,D).transpose(1,2)
        v=self.v_proj(x).view(B,T,KV,D).transpose(1,2)
        q=apply_rope(q,cos,sin); k=apply_rope(k,cos,sin)
        k=k.repeat_interleave(self.kv_repeat,dim=1); v=v.repeat_interleave(self.kv_repeat,dim=1)
        if key_valid is None:
            mask=None; causal=True
        else:
            qi=torch.arange(T,device=x.device)[:,None]
            kj=torch.arange(T,device=x.device)[None,:]
            mask=(kj<=qi)[None,None,:,:] & key_valid[:,None,None,:]
            mask=mask | (kj==qi)[None,None,:,:]
            causal=False
        y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=self.dropout if self.training else 0.0,is_causal=causal)
        return self.o_proj(y.transpose(1,2).contiguous().view(B,T,C))

class SwiGLU(nn.Module):
    def __init__(self,d_model,hidden_dim,dropout):
        super().__init__()
        self.gate_proj=nn.Linear(d_model,hidden_dim,bias=False)
        self.up_proj=nn.Linear(d_model,hidden_dim,bias=False)
        self.down_proj=nn.Linear(hidden_dim,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x))*self.up_proj(x)))

class EfficientDecoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1=RMSNorm(D_MODEL,RMS_EPS)
        self.attn=GQACausalSelfAttention(D_MODEL,N_HEADS,N_KV_HEADS,DROPOUT)
        self.norm2=RMSNorm(D_MODEL,RMS_EPS)
        self.ff=SwiGLU(D_MODEL,FFN_DIM,DROPOUT)
        self.resid_drop=nn.Dropout(DROPOUT)
    def forward(self,x,cos,sin,key_valid=None):
        x=x+self.resid_drop(self.attn(self.norm1(x),cos,sin,key_valid))
        x=x+self.ff(self.norm2(x))
        return x

class EfficientDecoderLM(nn.Module):
    def __init__(self):
        super().__init__(); self.vocab_size=VOCAB_SIZE; self.d_model=D_MODEL
        self.token_emb=nn.Embedding(VOCAB_SIZE,D_MODEL)
        self.drop=nn.Dropout(DROPOUT)
        self.rope=RotaryTable(D_MODEL//N_HEADS,MAX_POSITIONS,ROPE_BASE)
        self.blocks=nn.ModuleList([EfficientDecoderBlock() for _ in range(N_LAYERS)])
        self.final_norm=RMSNorm(D_MODEL,RMS_EPS)
    def forward_hidden(self,input_ids,key_valid=None):
        B,T=input_ids.shape; assert T<=MAX_POSITIONS
        pos=torch.arange(T,device=input_ids.device)[None,:].expand(B,T)
        cos,sin=self.rope(pos); x=self.drop(self.token_emb(input_ids))
        for block in self.blocks: x=block(x,cos,sin,key_valid)
        return self.final_norm(x)

def build_backbone():
    return EfficientDecoderLM()

bb=build_backbone()
obj=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=False)
state=obj["model"] if isinstance(obj,dict) and "model" in obj else obj
bb.load_state_dict(state,strict=True)
print("Strict Model4 load PASS | params:",sum(p.numel() for p in bb.parameters())/1e6,"M")


Strict Model4 load PASS | params: 99.30944 M


In [6]:
# CELL 6 — partial FT + hierarchical classifier
TRAIN_FROM=N_LAYERS-TRAIN_TOP_N_LAYERS
def configure_partial_ft(backbone):
    for p in backbone.parameters(): p.requires_grad=False
    for i in range(TRAIN_FROM,N_LAYERS):
        for p in backbone.blocks[i].parameters(): p.requires_grad=True
    for p in backbone.final_norm.parameters(): p.requires_grad=True

class HierarchicalCJPE(nn.Module):
    def __init__(self,backbone):
        super().__init__(); self.backbone=backbone
        self.chunk_proj=nn.Sequential(
            nn.Linear(2*D_MODEL,CHUNK_PROJ_DIM),nn.GELU(),
            nn.LayerNorm(CHUNK_PROJ_DIM),nn.Dropout(HEAD_DROPOUT)
        )
        self.chunk_pos=nn.Embedding(N_CHUNKS,CHUNK_PROJ_DIM)
        self.attn_score=nn.Sequential(nn.Linear(CHUNK_PROJ_DIM,128),nn.Tanh(),nn.Linear(128,1))
        self.fc=nn.Sequential(
            nn.Linear(2*CHUNK_PROJ_DIM,256),nn.GELU(),
            nn.Dropout(HEAD_DROPOUT),nn.Linear(256,2)
        )
    def forward(self,input_ids,valid,chunk_mask=None):
        B,K,T=input_ids.shape
        x=input_ids.view(B*K,T); m=valid.view(B*K,T)
        # Right-padded + causal => valid positions never attend to pads, so no padding mask is
        # needed. key_valid=None lets SDPA use the fast causal kernel (same outputs, faster).
        h=self.backbone.forward_hidden(x,key_valid=None)
        mf=m.unsqueeze(-1).to(h.dtype)
        mean=(h*mf).sum(1)/mf.sum(1).clamp(min=1)
        idx=m.long().sum(1).clamp(min=1)-1
        last=h[torch.arange(B*K,device=h.device),idx]
        z=self.chunk_proj(torch.cat([last.float(),mean.float()],-1)).view(B,K,-1)
        pos=torch.arange(K,device=z.device)[None,:]
        z=z+self.chunk_pos(pos)
        sc=self.attn_score(z).squeeze(-1)
        if chunk_mask is not None:          # test-time section ablation only
            cm=chunk_mask.to(z.device)[None,:]
            sc=sc.masked_fill(~cm,-1e4); zmax=z.masked_fill(~cm.unsqueeze(-1),-1e4)
        else:
            zmax=z
        a=torch.softmax(sc,-1)
        att=(z*a.unsqueeze(-1)).sum(1); mx=zmax.max(1).values
        return self.fc(torch.cat([att,mx],-1)),a

configure_partial_ft(bb)
model=HierarchicalCJPE(bb).to(DEVICE)
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
total=sum(p.numel() for p in model.parameters())
print("Train from block:",TRAIN_FROM)
print(f"Trainable {trainable/1e6:.2f}M / {total/1e6:.2f}M = {100*trainable/total:.1f}%")


Train from block: 12
Trainable 25.32M / 100.05M = 25.3%


In [7]:
# CELL 7 — tokenize/select HEAD-MIDDLE-TAIL (slice-based: no need to BPE-encode 400k-word cases)
SLICE_CHARS=CHUNK_LEN*8     # ~3k chars >> 384 BPE tokens

def _snap_forward(text,i):
    i=max(0,min(int(i),max(0,len(text)-1)))
    m=re.search(r"\s",text[i:i+400])
    return i+m.end() if m else i

def _enc(t):
    return tokenizer.encode(t)

def _window_start(text,center,width):
    s=max(0,int(center)-width//2)
    return _snap_forward(text,s)

def _last_evidence_position(text):
    low=text.lower()
    best=-1
    for cue in EVIDENCE_CUES:
        p=low.rfind(cue)
        if p>best:
            best=p
    return best

def select_chunks(text):
    """
    Four views of a long case:
      head / middle / evidence-or-late / tail.
    For short cases we derive all views from exact full BPE tokens.
    """
    n=len(text); W=SLICE_CHARS

    if n<=4*W:
        ids=_enc(text) or [tokenizer.unk_token_id]
        L=len(ids)

        if L<=CHUNK_LEN:
            ch=[ids,ids,ids,ids]
        else:
            def tok_window(frac):
                center=int(round(frac*(L-1)))
                s=max(0,min(L-CHUNK_LEN,center-CHUNK_LEN//2))
                return ids[s:s+CHUNK_LEN]

            # evidence view falls back to a late (75%) view for short docs.
            ch=[
                ids[:CHUNK_LEN],
                tok_window(0.50),
                tok_window(0.75),
                ids[-CHUNK_LEN:],
            ]
        est=L

    else:
        # Prefix
        head=_enc(text[:W])

        # True character-centred middle slice.
        ms=_window_start(text,n*0.50,W)
        mid=_enc(text[ms:ms+W])

        # Evidence chunk: centre around the LAST neutral disposition cue.
        # If absent, use a late-document (75%) region.
        ep=_last_evidence_position(text)
        ec=(ep if ep>=0 else int(n*0.75))
        es=_window_start(text,ec,W)
        evidence=_enc(text[es:es+W])

        # Suffix
        ts=_snap_forward(text,max(0,n-W))
        tail=_enc(text[ts:])

        if min(len(head),len(mid),len(evidence),len(tail))<CHUNK_LEN:
            # Rare token-density/pathological fallback: exact full tokenisation.
            ids=_enc(text) or [tokenizer.unk_token_id]
            L=len(ids)
            def tok_window(frac):
                center=int(round(frac*(L-1)))
                s=max(0,min(max(0,L-CHUNK_LEN),center-CHUNK_LEN//2))
                return ids[s:s+CHUNK_LEN]
            ch=[
                ids[:CHUNK_LEN],
                tok_window(0.50),
                tok_window(0.75),
                ids[-CHUNK_LEN:],
            ]
            est=L
        else:
            ch=[
                head[:CHUNK_LEN],
                mid[:CHUNK_LEN],
                evidence[:CHUNK_LEN],
                tail[-CHUNK_LEN:],
            ]
            est=int(n*len(head)/max(W,1))  # statistics only

    return [np.asarray(x,dtype=np.uint16) for x in ch],est

class CaseDataset(Dataset):
    def __init__(self,records,labeled):
        self.records=records; self.labeled=labeled; self.chunks=[]; self.full_lens=[]
        t0=time.time()
        for i,r in enumerate(records):
            ch,L=select_chunks(r["text"]); self.chunks.append(ch); self.full_lens.append(L)
            if (i+1)%5000==0: print(f"tokenized {i+1}/{len(records)} elapsed={time.time()-t0:.0f}s",flush=True)
        print(f"done {len(records)} in {time.time()-t0:.0f}s | est. median BPE={np.median(self.full_lens):.0f} p95={np.percentile(self.full_lens,95):.0f}")
    def __len__(self): return len(self.records)
    def __getitem__(self,i):
        r=self.records[i]; d={"id":r["id"],"chunks":self.chunks[i]}
        if self.labeled: d["label"]=int(r["label"])
        return d

def collate_cases(batch):
    B=len(batch)
    ids=torch.full((B,N_CHUNKS,CHUNK_LEN),tokenizer.pad_token_id,dtype=torch.long)
    valid=torch.zeros((B,N_CHUNKS,CHUNK_LEN),dtype=torch.bool)
    for b,x in enumerate(batch):
        for k,c in enumerate(x["chunks"]):
            z=torch.from_numpy(c.astype(np.int64)); n=min(len(z),CHUNK_LEN)
            ids[b,k,:n]=z[:n]; valid[b,k,:n]=True
    out={"input_ids":ids,"valid":valid,"ids":[x["id"] for x in batch]}
    if "label" in batch[0]:
        out["labels"]=torch.tensor([x["label"] for x in batch],dtype=torch.long)
    return out

train_ds=CaseDataset(train_records,True)
dev_ds=CaseDataset(clean_dev,True)
test_ds=CaseDataset(test_records,False)

g=torch.Generator(); g.manual_seed(SEED)
_kw=dict(collate_fn=collate_cases,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=="cuda",persistent_workers=NUM_WORKERS>0)
train_loader=DataLoader(train_ds,batch_size=MICRO_BATCH_SIZE,shuffle=True,generator=g,**_kw)
dev_loader=DataLoader(dev_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,**_kw)
test_loader=DataLoader(test_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,**_kw)

print("batches:",len(train_loader),len(dev_loader),len(test_loader))

# Verify the v2 fast causal-kernel optimisation on a real padded batch.
# Valid tokens in a causal model cannot attend to future right-padding.
@torch.no_grad()
def verify_fast_causal_equivalence():
    b=next(iter(dev_loader))
    x=b["input_ids"][:1].to(DEVICE)
    v=b["valid"][:1].to(DEVICE)
    B,K,T=x.shape
    xf=x.view(B*K,T); vf=v.view(B*K,T)
    model.backbone.eval()
    with torch.amp.autocast("cuda",dtype=torch.float16,enabled=(DEVICE.type=="cuda")):
        h_mask=model.backbone.forward_hidden(xf,key_valid=vf)
        h_fast=model.backbone.forward_hidden(xf,key_valid=None)
    d=(h_mask[vf]-h_fast[vf]).abs().max().item()
    print("fast causal-kernel valid-token max |Δ|:",d)
    assert d < 5e-2,(
        "Fast causal path deviates unexpectedly; restore key_valid masking."
    )
    model.train()

verify_fast_causal_equivalence()


tokenized 5000/32305 elapsed=17s
tokenized 10000/32305 elapsed=29s
tokenized 15000/32305 elapsed=43s
tokenized 20000/32305 elapsed=59s
tokenized 25000/32305 elapsed=73s
tokenized 30000/32305 elapsed=86s
done 32305 in 91s | est. median BPE=4416 p95=20119
done 994 in 3s | est. median BPE=6232 p95=21137
done 1517 in 5s | est. median BPE=6505 p95=22861
batches: 16153 166 253
fast causal-kernel valid-token max |Δ|: 0.0


In [8]:
# CELL 8 — optimizer, metric, threshold search
counts=Counter(r["label"] for r in train_records)
class_w=torch.tensor([len(train_records)/(2*counts[i]) for i in [0,1]],dtype=torch.float32,device=DEVICE)
class_w/=class_w.mean()
print("class weights:",class_w.tolist())

def make_groups(m):
    gs=[]
    for i in range(TRAIN_FROM,N_LAYERS):
        depth=(N_LAYERS-1)-i
        gs.append({"params":[p for p in m.backbone.blocks[i].parameters() if p.requires_grad],
                   "lr":BACKBONE_LR*(LAYERWISE_LR_DECAY**depth),"weight_decay":WEIGHT_DECAY,"name":f"block_{i}"})
    gs.append({"params":list(m.backbone.final_norm.parameters()),"lr":BACKBONE_LR,"weight_decay":0.0,"name":"final_norm"})
    hp=[p for n,p in m.named_parameters() if p.requires_grad and not n.startswith("backbone.")]
    gs.append({"params":hp,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY,"name":"head"})
    return gs

groups=make_groups(model)
try: optimizer=torch.optim.AdamW(groups,fused=DEVICE.type=="cuda")
except Exception: optimizer=torch.optim.AdamW(groups)

updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
total_updates=NUM_EPOCHS*updates_per_epoch
warmup=max(1,int(total_updates*WARMUP_RATIO))
def lr_mult(step):
    if step<warmup: return max(1e-8,step/warmup)
    p=(step-warmup)/max(1,total_updates-warmup)
    return 0.5*(1+math.cos(math.pi*min(max(p,0),1)))
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_mult)

amp_enabled=USE_AMP and DEVICE.type=="cuda"
scaler=torch.amp.GradScaler("cuda",enabled=amp_enabled,init_scale=AMP_INIT_SCALE,growth_interval=2000)

def macro_f1(y,p):
    return f1_score(y,p,average="macro",labels=[0,1],zero_division=0)

def search_threshold(y,prob,smooth=THRESH_SMOOTH):
    """
    Robust operating point on the 994-case DEV set.
    We smooth the exact Macro-F1 curve with EDGE padding (not zero padding),
    then choose the smoothed optimum. We also return the absolute best-grid F1
    for diagnostics/checkpoint auditing.
    """
    fs=np.array(
        [macro_f1(y,(prob>=t).astype(np.int64)) for t in THRESHOLDS],
        dtype=np.float64,
    )
    k=max(1,int(smooth))
    if k>1:
        if k%2==0: k+=1
        pad=k//2
        sm=np.convolve(
            np.pad(fs,(pad,pad),mode="edge"),
            np.ones(k,dtype=np.float64)/k,
            mode="valid",
        )
    else:
        sm=fs.copy()

    i=int(np.argmax(sm))
    t=float(THRESHOLDS[i])
    pred=(prob>=t).astype(np.int64)
    return float(fs[i]),t,pred,float(fs.max())


@torch.no_grad()
def predict_logits(m,loader):
    m.eval(); lgs=[]; ys=[]; ids=[]; ats=[]
    for b in loader:
        x=b["input_ids"].to(DEVICE,non_blocking=True); v=b["valid"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
            lg,a=m(x,v)
        lgs.append(lg.float().cpu().numpy()); ats.append(a.float().cpu().numpy()); ids.extend(b["ids"])
        if "labels" in b: ys.append(b["labels"].numpy())
    return np.concatenate(lgs),(np.concatenate(ys) if ys else None),ids,np.concatenate(ats)

def _p1(lg):
    lg=lg.astype(np.float64); lg=lg-lg.max(-1,keepdims=True); e=np.exp(lg)
    return (e[:,1]/e.sum(-1)).astype(np.float32)

def predict_probs(m,loader):
    lg,y,ids,att=predict_logits(m,loader)
    return _p1(lg),y,ids,att


class weights: [0.8286643028259277, 1.1713356971740723]


In [9]:
# CELL 9 — train / select by clean DEV Macro-F1
BEST=ROOT/"best_partialft_cjpe.pt"
history=[]; best_f1=-1.; best_thr=.5; best_frac=None; bad=0; global_update=0; overflows=0; top_snapshots=[]
optimizer.zero_grad(set_to_none=True)

eval_steps=[]
for j in range(1,EVAL_PER_EPOCH+1):
    s=int(round(len(train_loader)*j/EVAL_PER_EPOCH))
    if s<len(train_loader): s=max(GRAD_ACCUM_STEPS,(s//GRAD_ACCUM_STEPS)*GRAD_ACCUM_STEPS)
    else: s=len(train_loader)
    eval_steps.append(s)
eval_steps=sorted(set(eval_steps))
print("DEV eval batches:",eval_steps)

stop=False; train_start=time.time()
for epoch in range(1,NUM_EPOCHS+1):
    model.train(); run=0.; nr=0; ep0=time.time()
    for step,b in enumerate(train_loader,1):
        x=b["input_ids"].to(DEVICE,non_blocking=True); v=b["valid"].to(DEVICE,non_blocking=True); y=b["labels"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
            lg,_=model(x,v)
            raw=F.cross_entropy(lg.float(),y,weight=class_w,label_smoothing=LABEL_SMOOTHING)
            loss=raw/GRAD_ACCUM_STEPS
        assert torch.isfinite(loss).item()
        scaler.scale(loss).backward(); run+=float(raw.detach().cpu()); nr+=1

        if step%GRAD_ACCUM_STEPS==0 or step==len(train_loader):
            scaler.unscale_(optimizer)
            gn=torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],GRAD_CLIP)
            finite=bool(torch.isfinite(torch.as_tensor(gn)).item())
            old=float(scaler.get_scale()); scaler.step(optimizer); scaler.update(); new=float(scaler.get_scale())
            optimizer.zero_grad(set_to_none=True)
            if finite and new>=old:
                scheduler.step(); global_update+=1
            else:
                overflows+=1; print(f"⚠️ AMP overflow handled batch={step} scale {old:.0f}->{new:.0f}")

        if step%500==0 or step==len(train_loader):
            elapsed=time.time()-ep0; rate=elapsed/max(step,1); eta=rate*(len(train_loader)-step)
            print(f"epoch={epoch} batch={step}/{len(train_loader)} loss={run/max(nr,1):.4f} elapsed={elapsed/60:.1f}m ETA_epoch={eta/60:.1f}m",flush=True)

        if step in eval_steps:
            prob,ydev,idsdev,att=predict_probs(model,dev_loader)
            f,t,pred,grid_best_f1=search_threshold(ydev,prob)
            row={"epoch":epoch,"epoch_frac":(epoch-1)+step/len(train_loader),"batch":step,
                 "macro_f1":f,"threshold":t,"grid_best_macro_f1":grid_best_f1,"f1_at_0.5":float(macro_f1(ydev,(prob>=0.5).astype(np.int64))),"train_loss":run/max(nr,1),
                 "global_update":global_update,"amp_overflows":overflows,
                 "attention_mean":att.mean(0).tolist()}
            history.append(row); print("DEV:",row)
            # Retain top-k heldout snapshots for a same-method test ensemble.
            qualifies=(
                len(top_snapshots)<TOPK_SNAPSHOTS
                or f>min(x["f1"] for x in top_snapshots)
            )
            state_now=None
            if qualifies:
                state_now={k:v.detach().cpu() for k,v in model.state_dict().items()}
                sp=ROOT/f"partialft_snapshot_e{row['epoch_frac']:.3f}_f1_{f:.6f}.pt"
                torch.save({
                    "model":state_now,
                    "macro_f1":float(f),
                    "threshold":float(t),
                    "epoch_frac":float(row["epoch_frac"]),
                },sp)
                top_snapshots.append({
                    "f1":float(f),"threshold":float(t),
                    "epoch_frac":float(row["epoch_frac"]),"path":str(sp)
                })
                top_snapshots.sort(key=lambda x:x["f1"],reverse=True)
                while len(top_snapshots)>TOPK_SNAPSHOTS:
                    drop=top_snapshots.pop(-1)
                    try: os.remove(drop["path"])
                    except FileNotFoundError: pass
                print("Top snapshots:",[(round(x["f1"],5),round(x["epoch_frac"],3)) for x in top_snapshots])

            if f>best_f1+1e-12:
                best_f1=f; best_thr=t; best_frac=row["epoch_frac"]; bad=0
                torch.save({"model":(state_now if state_now is not None else {k:v.detach().cpu() for k,v in model.state_dict().items()}),
                            "best_macro_f1":best_f1,"best_threshold":best_thr,
                            "best_epoch_frac":best_frac,"history":history},BEST)
                print(f"✅ NEW BEST F1={best_f1:.6f} threshold={best_thr:.3f}")
            else: bad+=1
            if bad>=PATIENCE_EVALS:
                stop=True; print("Early stopping."); break
            model.train()
    if stop: break

json.dump(history,open(ROOT/"history.json","w"),indent=2)
json.dump(top_snapshots,open(ROOT/"top_snapshots.json","w"),indent=2)
assert BEST.is_file()
print("DONE best F1/thr/frac:",best_f1,best_thr,best_frac)


DEV eval batches: [5384, 10768, 16153]
epoch=1 batch=500/16153 loss=0.7056 elapsed=0.8m ETA_epoch=25.5m
epoch=1 batch=1000/16153 loss=0.7083 elapsed=1.7m ETA_epoch=25.4m
epoch=1 batch=1500/16153 loss=0.7129 elapsed=2.5m ETA_epoch=24.6m
epoch=1 batch=2000/16153 loss=0.7095 elapsed=3.4m ETA_epoch=23.8m
epoch=1 batch=2500/16153 loss=0.7087 elapsed=4.2m ETA_epoch=23.0m
epoch=1 batch=3000/16153 loss=0.7081 elapsed=5.1m ETA_epoch=22.2m
epoch=1 batch=3500/16153 loss=0.7064 elapsed=5.9m ETA_epoch=21.3m
epoch=1 batch=4000/16153 loss=0.7006 elapsed=6.7m ETA_epoch=20.5m
epoch=1 batch=4500/16153 loss=0.6946 elapsed=7.6m ETA_epoch=19.7m
epoch=1 batch=5000/16153 loss=0.6905 elapsed=8.4m ETA_epoch=18.8m
DEV: {'epoch': 1, 'epoch_frac': 0.333312697331765, 'batch': 5384, 'macro_f1': 0.6034359224881941, 'threshold': 0.29, 'grid_best_macro_f1': 0.607645875251509, 'f1_at_0.5': 0.44626451062892225, 'train_loss': 0.685450034448531, 'global_update': 673, 'amp_overflows': 0, 'attention_mean': [0.06598684936761

In [10]:
# CELL 10 — exact Phase-4 evaluator
best_obj=torch.load(BEST,map_location="cpu",weights_only=False)
model.load_state_dict(best_obj["model"],strict=True)
best_thr=float(best_obj["best_threshold"])

dev_prob,dev_y,dev_ids,dev_att=predict_probs(model,dev_loader)
dev_pred=(dev_prob>=best_thr).astype(np.int64)
truth=[{"id":str(i),"gold":int(y)} for i,y in zip(dev_ids,dev_y)]
pred=[{"id":str(i),"pred":int(p)} for i,p in zip(dev_ids,dev_pred)]

TRUTH=ROOT/"truth_dev.json"; PRED=ROOT/"prediction_dev.json"
json.dump(truth,open(TRUTH,"w"),indent=2); json.dump(pred,open(PRED,"w"),indent=2)

EVALUATOR_CODE=r'''#!/usr/bin/env python3
import argparse,json,sys
from sklearn.metrics import f1_score,precision_score,recall_score
def read_truth(path):
    try: x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e: raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or isinstance(r['gold'],bool) or r['gold'] not in (0,1): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        out[r['id']]=r['gold']
    return out
def read_pred(path):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or isinstance(r['pred'],bool) or r['pred'] not in (0,1): raise ValueError(f'Invalid prediction item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
        out[r['id']]=r['pred']
    return out
def main():
    ap=argparse.ArgumentParser(); ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t=read_truth(a.truth); p=read_pred(a.prediction)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=[t[i] for i in t]; yp=[p[i] for i in t]
    print(f'Examples: {len(t)}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",labels=[0,1],zero_division=0):.6f}')
    print(f'Precision: {precision_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
    print(f'Recall: {recall_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)
'''
EVAL=ROOT/"evaluate_phase4.py"; EVAL.write_text(EVALUATOR_CODE,encoding="utf-8")
res=subprocess.run([sys.executable,str(EVAL),"--truth",str(TRUTH),"--prediction",str(PRED)],capture_output=True,text=True)
print(res.stdout)
if res.stderr: print(res.stderr)
assert res.returncode==0
print("✅ Exact Phase-4 evaluator PASS")

np.savez_compressed(ROOT/"dev_probs.npz",prob=dev_prob.astype(np.float32),y=dev_y.astype(np.int64),
                    id=np.asarray(dev_ids,dtype=object),threshold=np.asarray([best_thr],dtype=np.float32),
                    chunk_attention=dev_att.astype(np.float32))


Examples: 994
Macro-F1: 0.676710
Precision: 0.716019
Recall: 0.593561

✅ Exact Phase-4 evaluator PASS


In [11]:
# CELL 10b — section ablation + error analysis on clean DEV (inference only; for the report)
@torch.no_grad()
def probs_with_mask(m,loader,mask):
    m.eval(); ps=[]; cm=torch.tensor(mask,dtype=torch.bool,device=DEVICE)
    for b in loader:
        x=b["input_ids"].to(DEVICE,non_blocking=True); v=b["valid"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
            lg,_=m(x,v,chunk_mask=cm)
        ps.append(torch.softmax(lg.float(),-1)[:,1].cpu().numpy())
    return np.concatenate(ps)

subsets={
 "all":[1,1,1,1],
 "head only":[1,0,0,0],
 "middle only":[0,1,0,0],
 "evidence only":[0,0,1,0],
 "tail only":[0,0,0,1],
 "no head":[0,1,1,1],
 "no middle":[1,0,1,1],
 "no evidence":[1,1,0,1],
 "no tail":[1,1,1,0],
}
ablation={}
print("TEST-TIME section ablation of the trained model (chunk aggregation masked; DEV Macro-F1):")
for name,mask in subsets.items():
    p=probs_with_mask(model,dev_loader,mask); f,t,_,gridbest=search_threshold(dev_y,p)
    f05=float(macro_f1(dev_y,(p>=0.5).astype(np.int64)))
    ablation[name]={"macro_f1_tuned":f,"threshold":t,"grid_best_macro_f1":gridbest,"macro_f1_at_0.5":f05}
    print(f"  {name:24s} F1(tuned thr={t:.3f})={f:.4f} | F1@0.5={f05:.4f}")
json.dump(ablation,open(ROOT/"section_ablation_dev.json","w"),indent=2)

L=np.asarray(dev_ds.full_lens); y=np.asarray(dev_y); pr=np.asarray(dev_pred)
fp=(pr==1)&(y==0); fn=(pr==0)&(y==1); ok=pr==y
print(f"\nDEV errors: FP={fp.sum()} FN={fn.sum()} correct={ok.sum()} of {len(y)}")
for nm,mk_ in [("correct",ok),("false positive",fp),("false negative",fn)]:
    if mk_.sum(): print(f"  {nm:15s} n={mk_.sum():4d} | median est. BPE length={np.median(L[mk_]):.0f} | mean P(class1)={dev_prob[mk_].mean():.3f}")
for lo,hi in [(0,1500),(1500,3000),(3000,6000),(6000,10**9)]:
    mk_=(L>=lo)&(L<hi)
    if mk_.sum(): print(f"  length {lo:5d}-{hi if hi<10**9 else 'inf'}: n={mk_.sum():4d} accuracy={ok[mk_].mean():.3f}")
json.dump([{"id":i,"gold":int(g),"pred":int(p),"prob1":float(q),"est_bpe_len":int(l)} for i,g,p,q,l in zip(dev_ids,y,pr,dev_prob,L) if g!=p],
          open(ROOT/"dev_errors.json","w"),indent=1)


TEST-TIME section ablation of the trained model (chunk aggregation masked; DEV Macro-F1):
  all                      F1(tuned thr=0.425)=0.6767 | F1@0.5=0.6278
  head only                F1(tuned thr=0.445)=0.5665 | F1@0.5=0.5370
  middle only              F1(tuned thr=0.555)=0.5714 | F1@0.5=0.4974
  evidence only            F1(tuned thr=0.480)=0.5634 | F1@0.5=0.5400
  tail only                F1(tuned thr=0.475)=0.6382 | F1@0.5=0.6319
  no head                  F1(tuned thr=0.425)=0.6686 | F1@0.5=0.6254
  no middle                F1(tuned thr=0.405)=0.6748 | F1@0.5=0.6242
  no evidence              F1(tuned thr=0.440)=0.6632 | F1@0.5=0.6407
  no tail                  F1(tuned thr=0.450)=0.6275 | F1@0.5=0.5799

DEV errors: FP=117 FN=202 correct=675 of 994
  correct         n= 675 | median est. BPE length=6153 | mean P(class1)=0.457
  false positive  n= 117 | median est. BPE length=6458 | mean P(class1)=0.526
  false negative  n= 202 | median est. BPE length=6468 | mean P(class1)=0.340


In [12]:
# CELL 11 — fresh model helper + clean refit (reuses the already-tokenised train/dev datasets)
def fresh_model():
    b=build_backbone()
    o=torch.load(PHASE1_CKPT,map_location="cpu",weights_only=False)
    st=o["model"] if isinstance(o,dict) and "model" in o else o
    b.load_state_dict(st,strict=True); configure_partial_ft(b)
    return HierarchicalCJPE(b).to(DEVICE)

refit_model=None
if RUN_REFIT:
    all_records=train_records+clean_dev
    all_ds=torch.utils.data.ConcatDataset([train_ds,dev_ds])
    gg=torch.Generator(); gg.manual_seed(SEED)
    all_loader=DataLoader(all_ds,batch_size=MICRO_BATCH_SIZE,shuffle=True,generator=gg,collate_fn=collate_cases,
                          num_workers=NUM_WORKERS,pin_memory=DEVICE.type=="cuda",persistent_workers=NUM_WORKERS>0)
    refit_model=fresh_model()
    rgs=make_groups(refit_model)
    try: ropt=torch.optim.AdamW(rgs,fused=DEVICE.type=="cuda")
    except Exception: ropt=torch.optim.AdamW(rgs)

    target_batches=max(GRAD_ACCUM_STEPS,int(round(float(best_obj["best_epoch_frac"])*len(all_loader)/GRAD_ACCUM_STEPS))*GRAD_ACCUM_STEPS)
    upe_all=math.ceil(len(all_loader)/GRAD_ACCUM_STEPS)
    horizon_ru=(NUM_EPOCHS*upe_all) if REFIT_MATCH_ORIGINAL_SCHEDULE else math.ceil(target_batches/GRAD_ACCUM_STEPS)
    rw=max(1,int(horizon_ru*WARMUP_RATIO))
    def rmult(step):
        if step<rw: return max(1e-8,step/rw)
        p=(step-rw)/max(1,horizon_ru-rw); return 0.5*(1+math.cos(math.pi*min(max(p,0),1)))
    rsched=torch.optim.lr_scheduler.LambdaLR(ropt,rmult)
    rscaler=torch.amp.GradScaler("cuda",enabled=amp_enabled,init_scale=AMP_INIT_SCALE,growth_interval=2000)
    ac=Counter(r["label"] for r in all_records)
    cw=torch.tensor([len(all_records)/(2*ac[i]) for i in [0,1]],dtype=torch.float32,device=DEVICE); cw/=cw.mean()
    print("refit target micro-batches:",target_batches,"| LR horizon (updates):",horizon_ru,"| replay original schedule:",REFIT_MATCH_ORIGINAL_SCHEDULE)

    ropt.zero_grad(set_to_none=True); seen=0; over=0; t0=time.time()
    while seen<target_batches:
        refit_model.train()
        for b in all_loader:
            if seen>=target_batches: break
            x=b["input_ids"].to(DEVICE,non_blocking=True); v=b["valid"].to(DEVICE,non_blocking=True); y=b["labels"].to(DEVICE,non_blocking=True)
            with torch.amp.autocast("cuda",dtype=torch.float16,enabled=amp_enabled):
                lg,_=refit_model(x,v)
                raw=F.cross_entropy(lg.float(),y,weight=cw,label_smoothing=LABEL_SMOOTHING)
                loss=raw/GRAD_ACCUM_STEPS
            rscaler.scale(loss).backward(); seen+=1
            if seen%GRAD_ACCUM_STEPS==0 or seen>=target_batches:
                rscaler.unscale_(ropt)
                gn=torch.nn.utils.clip_grad_norm_([p for p in refit_model.parameters() if p.requires_grad],GRAD_CLIP)
                finite=bool(torch.isfinite(torch.as_tensor(gn)).item())
                old=float(rscaler.get_scale()); rscaler.step(ropt); rscaler.update(); new=float(rscaler.get_scale())
                ropt.zero_grad(set_to_none=True)
                if finite and new>=old: rsched.step()
                else: over+=1
            if seen%500==0 or seen>=target_batches:
                rate=(time.time()-t0)/seen
                print(f"refit {seen}/{target_batches} ETA={(target_batches-seen)*rate/60:.1f}m overflow={over}",flush=True)

    torch.save({"model":{k:v.detach().cpu() for k,v in refit_model.state_dict().items()},
                "source_best_epoch_frac":best_obj["best_epoch_frac"],"threshold":best_thr,"overflows":over},
               ROOT/"final_refit_partialft_cjpe.pt")
    print("✅ refit complete")
else:
    print("RUN_REFIT=False")


refit target micro-batches: 16648 | LR horizon (updates): 4164 | replay original schedule: True
refit 500/16648 ETA=27.4m overflow=0
refit 1000/16648 ETA=26.5m overflow=0
refit 1500/16648 ETA=25.6m overflow=0
refit 2000/16648 ETA=24.7m overflow=0
refit 2500/16648 ETA=23.9m overflow=0
refit 3000/16648 ETA=23.1m overflow=0
refit 3500/16648 ETA=22.2m overflow=0
refit 4000/16648 ETA=21.4m overflow=0
refit 4500/16648 ETA=20.5m overflow=0
refit 5000/16648 ETA=19.7m overflow=0
refit 5500/16648 ETA=18.8m overflow=0
refit 6000/16648 ETA=18.0m overflow=0
refit 6500/16648 ETA=17.2m overflow=0
refit 7000/16648 ETA=16.3m overflow=0
refit 7500/16648 ETA=15.5m overflow=0
refit 8000/16648 ETA=14.6m overflow=0
refit 8500/16648 ETA=13.8m overflow=0
refit 9000/16648 ETA=12.9m overflow=0
refit 9500/16648 ETA=12.1m overflow=0
refit 10000/16648 ETA=11.2m overflow=0
refit 10500/16648 ETA=10.4m overflow=0
refit 11000/16648 ETA=9.5m overflow=0
refit 11500/16648 ETA=8.7m overflow=0
refit 12000/16648 ETA=7.9m ov

In [13]:
# CELL 12 — test inference (no test labels) + label-free operating-point calibration
def _logit(p): p=np.clip(p.astype(np.float64),1e-6,1-1e-6); return np.log(p/(1-p))

held_lg,_,held_ids,held_att=predict_logits(model,test_loader)
assert held_ids==[r["id"] for r in test_records] and len(held_lg)==1517
held_prob=_p1(held_lg)

refit_prob=None; refit_lg=None; refit_thr_rate=None
if refit_model is not None:
    refit_lg,_,rid,ratt=predict_logits(refit_model,test_loader)
    assert rid==held_ids
    refit_prob=_p1(refit_lg)
    # The refit model saw the dev set, so the dev-tuned threshold cannot be re-checked for it.
    # Calibrate it WITHOUT labels: choose the threshold at which it predicts the same positive
    # rate on the test inputs as the validated held-out model does at its dev-tuned threshold.
    held_rate=float((held_prob>=best_thr).mean())
    refit_thr_rate=float(np.clip(np.quantile(refit_prob,1.0-held_rate),0.05,0.95))
    print(f"held-out positive rate @thr={best_thr:.3f}: {held_rate:.3f} | refit raw-thr rate: {(refit_prob>=best_thr).mean():.3f} | rate-matched refit thr: {refit_thr_rate:.3f}")

np.savez_compressed(ROOT/"test_probs.npz",heldout=held_prob.astype(np.float32),
                    refit=(refit_prob.astype(np.float32) if refit_prob is not None else np.array([],dtype=np.float32)),
                    heldout_logits=held_lg.astype(np.float32),
                    refit_logits=(refit_lg.astype(np.float32) if refit_lg is not None else np.array([],dtype=np.float32)),
                    id=np.asarray(held_ids,dtype=object),
                    threshold=np.asarray([best_thr],dtype=np.float32),
                    refit_rate_matched_threshold=np.asarray([refit_thr_rate if refit_thr_rate is not None else np.nan],dtype=np.float32),
                    heldout_chunk_attention=held_att.astype(np.float32))
print("heldout probability mean:",held_prob.mean())
if refit_prob is not None: print("refit probability mean:",refit_prob.mean())


# Top-k heldout snapshot probability ensemble.
snapshot_prob=None
snapshot_threshold=None
snapshot_member_probs=[]
if MAKE_SNAPSHOT_ENSEMBLES and (ROOT/"top_snapshots.json").is_file():
    top_meta=json.load(open(ROOT/"top_snapshots.json",encoding="utf-8"))[:TOPK_SNAPSHOTS]
    snapshot_thresholds=[]
    for info in top_meta:
        sp=Path(info["path"])
        if not sp.is_file():
            print("skip missing snapshot:",sp)
            continue
        sm=fresh_model()
        so=torch.load(sp,map_location="cpu",weights_only=False)
        sm.load_state_dict(so["model"],strict=True)
        slg,_,sids,_=predict_logits(sm,test_loader)
        assert sids==held_ids
        snapshot_member_probs.append(_p1(slg))
        snapshot_thresholds.append(float(info["threshold"]))
        del sm,so,slg
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    if len(snapshot_member_probs)>=2:
        # Average calibrated margins rather than raw probabilities because each
        # heldout snapshot has its own DEV-selected operating threshold.
        margins=[
            _logit(p)-_logit(np.array([t]))
            for p,t in zip(snapshot_member_probs,snapshot_thresholds)
        ]
        snapshot_margin=np.mean(np.stack(margins,axis=0),axis=0)
        snapshot_prob=(1.0/(1.0+np.exp(-snapshot_margin))).astype(np.float32)
        snapshot_threshold=0.5
        print(
            "snapshot ensemble members:",len(snapshot_member_probs),
            "| positive rate:",float((snapshot_prob>=0.5).mean())
        )


held-out positive rate @thr=0.425: 0.439 | refit raw-thr rate: 0.621 | rate-matched refit thr: 0.448
heldout probability mean: 0.45398533
refit probability mean: 0.460179
snapshot ensemble members: 3 | positive rate: 0.4878048780487805


In [14]:
# CELL 13 — Codabench variants
official_ids=[r["id"] for r in test_records]
assert len(official_ids)==len(set(official_ids))==1517

def make_pred_from_labels(labels):
    return [{"id":str(cid),"pred":int(l)} for cid,l in zip(official_ids,labels)]

def validate(pred):
    assert isinstance(pred,list) and len(pred)==len(official_ids)
    ids=[]
    for r in pred:
        assert set(r)=={"id","pred"} and isinstance(r["id"],str) and r["id"]
        assert type(r["pred"]) is int and r["pred"] in (0,1)
        ids.append(r["id"])
    assert len(ids)==len(set(ids)) and set(ids)==set(official_ids)

var_labels={"heldout":(held_prob>=best_thr).astype(int)}
if snapshot_prob is not None:
    var_labels["snapshot_ensemble"]=(snapshot_prob>=0.5).astype(int)
if refit_prob is not None:
    var_labels["refit"]=(refit_prob>=refit_thr_rate).astype(int)            # rate-matched threshold
    var_labels["refit_rawthr"]=(refit_prob>=best_thr).astype(int)           # dev-tuned threshold as-is
    if ENSEMBLE_HELDOUT_REFIT:
        # average each model's margin relative to its OWN operating threshold
        margin=0.5*(_logit(held_prob)-_logit(np.array([best_thr])))+0.5*(_logit(refit_prob)-_logit(np.array([refit_thr_rate])))
        var_labels["ensemble"]=(margin>=0).astype(int)

        if snapshot_prob is not None:
            # snapshot_prob is already a sigmoid of averaged calibrated margins.
            smargin=_logit(snapshot_prob)
            rmargin=_logit(refit_prob)-_logit(np.array([refit_thr_rate]))
            var_labels["snapshot_refit_ensemble"]=(0.5*smargin+0.5*rmargin>=0).astype(int)

for name,lab in var_labels.items():
    pred=make_pred_from_labels(lab); validate(pred)
    jp=ROOT/f"prediction_{name}.json"; zp=ROOT/f"submission_{name}.zip"
    json.dump(pred,open(jp,"w"),indent=2)
    with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(jp,arcname="prediction.json")
    with zipfile.ZipFile(zp,"r") as z: assert z.namelist()==["prediction.json"]
    print(f"{name:13s} positive rate={lab.mean():.3f}",Counter(lab.tolist()),zp.name)
names=list(var_labels)
for i in range(len(names)):
    for j in range(i+1,len(names)):
        print(f"agreement {names[i]} vs {names[j]}: {np.mean(var_labels[names[i]]==var_labels[names[j]]):.4f}")

default="ensemble" if "ensemble" in var_labels else "heldout"
shutil.copy2(ROOT/f"submission_{default}.zip",ROOT/"submission.zip")
print("✅ submission.zip =",default)


heldout       positive rate=0.439 Counter({0: 851, 1: 666}) submission_heldout.zip
snapshot_ensemble positive rate=0.488 Counter({0: 777, 1: 740}) submission_snapshot_ensemble.zip
refit         positive rate=0.439 Counter({0: 851, 1: 666}) submission_refit.zip
refit_rawthr  positive rate=0.621 Counter({1: 942, 0: 575}) submission_refit_rawthr.zip
ensemble      positive rate=0.430 Counter({0: 864, 1: 653}) submission_ensemble.zip
snapshot_refit_ensemble positive rate=0.476 Counter({0: 795, 1: 722}) submission_snapshot_refit_ensemble.zip
agreement heldout vs snapshot_ensemble: 0.8945
agreement heldout vs refit: 0.8497
agreement heldout vs refit_rawthr: 0.7614
agreement heldout vs ensemble: 0.9611
agreement heldout vs snapshot_refit_ensemble: 0.9038
agreement snapshot_ensemble vs refit: 0.8405
agreement snapshot_ensemble vs refit_rawthr: 0.8075
agreement snapshot_ensemble vs ensemble: 0.9018
agreement snapshot_ensemble vs snapshot_refit_ensemble: 0.9644
agreement refit vs refit_rawthr: 0.

In [15]:
# CELL 14 — manifest
manifest={
 "phase":4,"experiment":1,"method":"Partial/layer-wise fine-tuning",
 "phase1_model":"Model4 efficient decoder",
 "checkpoint":str(PHASE1_CKPT),"tokenizer_sha256":sha256_file(TOKENIZER_FILE),
 "train_unique":len(train_records),"clean_dev":len(clean_dev),"test":len(test_records),
 "excluded_dev_test_overlap":overlap_count,
 "chunks":CHUNK_NAMES,"chunk_len":CHUNK_LEN,
 "evidence_selector":"last neutral disposition cue; fallback late-75pct",
 "topk_snapshots":TOPK_SNAPSHOTS,
 "train_top_n_layers":TRAIN_TOP_N_LAYERS,"train_from_block":TRAIN_FROM,
 "backbone_lr":BACKBONE_LR,"head_lr":HEAD_LR,
 "best_dev_macro_f1":float(best_obj["best_macro_f1"]),
 "best_threshold":float(best_obj["best_threshold"]),
 "best_epoch_frac":float(best_obj["best_epoch_frac"]),
 "lowercase":bool(LOWERCASE),"collapse_ws":bool(COLLAPSE_WS),"thresh_smooth":THRESH_SMOOTH,
 "refit_match_original_schedule":bool(REFIT_MATCH_ORIGINAL_SCHEDULE),"refit_rate_matched_threshold":(None if refit_thr_rate is None else float(refit_thr_rate)),
 "test_labels_used":False,"overlap_dev_labels_used":False,
}
json.dump(manifest,open(ROOT/"experiment_manifest.json","w"),indent=2)
print(json.dumps(manifest,indent=2))


{
  "phase": 4,
  "experiment": 1,
  "method": "Partial/layer-wise fine-tuning",
  "phase1_model": "Model4 efficient decoder",
  "checkpoint": "/kaggle/input/datasets/aaryant23/phase2-model4/eff_full_best (6).pt",
  "tokenizer_sha256": "9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d",
  "train_unique": 32305,
  "clean_dev": 994,
  "test": 1517,
  "excluded_dev_test_overlap": 1517,
  "chunks": [
    "head",
    "middle",
    "evidence",
    "tail"
  ],
  "chunk_len": 384,
  "evidence_selector": "last neutral disposition cue; fallback late-75pct",
  "topk_snapshots": 3,
  "train_top_n_layers": 5,
  "train_from_block": 12,
  "backbone_lr": 3e-05,
  "head_lr": 0.0005,
  "best_dev_macro_f1": 0.6767103957847129,
  "best_threshold": 0.425,
  "best_epoch_frac": 1.0,
  "lowercase": true,
  "collapse_ws": true,
  "thresh_smooth": 9,
  "refit_match_original_schedule": true,
  "refit_rate_matched_threshold": 0.44819948077201843,
  "test_labels_used": false,
  "overlap_dev_labels_

In [16]:
# CELL 14b — export teacher logits for the knowledge-distillation experiment (runs AFTER submissions exist)
if EXPORT_TEACHER_LOGITS:
    _kw2=dict(collate_fn=collate_cases,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=="cuda")
    tr_loader=DataLoader(train_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,**_kw2)
    tr_lg,tr_y,tr_ids,_=predict_logits(model,tr_loader)
    dv_lg,dv_y,dv_ids,_=predict_logits(model,dev_loader)
    np.savez_compressed(ROOT/"teacher_logits_cjpe.npz",
        train_logits=tr_lg.astype(np.float32),train_y=tr_y.astype(np.int64),train_id=np.asarray(tr_ids,dtype=object),
        dev_logits=dv_lg.astype(np.float32),dev_y=dv_y.astype(np.int64),dev_id=np.asarray(dv_ids,dtype=object),
        test_heldout_logits=held_lg.astype(np.float32),
        test_refit_logits=(refit_lg.astype(np.float32) if refit_lg is not None else np.array([],dtype=np.float32)),
        test_id=np.asarray(held_ids,dtype=object),
        threshold=np.asarray([best_thr],dtype=np.float32))
    print("✅ teacher_logits_cjpe.npz | train acc of held-out teacher:",round(float((tr_lg.argmax(-1)==tr_y).mean()),4))
else:
    print("EXPORT_TEACHER_LOGITS=False")


    if refit_model is not None:
        # Final teacher signal for a future KD refit: refit teacher on TRAIN+CLEAN-DEV.
        all_eval_ds=torch.utils.data.ConcatDataset([train_ds,dev_ds])
        all_eval_loader=DataLoader(
            all_eval_ds,batch_size=EVAL_BATCH_SIZE,shuffle=False,
            collate_fn=collate_cases,num_workers=NUM_WORKERS,
            pin_memory=DEVICE.type=="cuda"
        )
        rf_lg,rf_y,rf_ids,_=predict_logits(refit_model,all_eval_loader)
        np.savez_compressed(
            ROOT/"teacher_logits_cjpe_refit_train_dev.npz",
            logits=rf_lg.astype(np.float32),
            y=rf_y.astype(np.int64),
            id=np.asarray(rf_ids,dtype=object),
        )
        print(
            "✅ teacher_logits_cjpe_refit_train_dev.npz",
            "| rows:",len(rf_ids),
            "| train+clean-dev refit teacher acc:",
            round(float((rf_lg.argmax(-1)==rf_y).mean()),4)
        )


KeyboardInterrupt: 

## Outputs to keep

- completed notebook
- `best_partialft_cjpe.pt`
- `final_refit_partialft_cjpe.pt`
- `history.json`
- `dev_probs.npz`
- `test_probs.npz`
- `experiment_manifest.json`
- submission ZIPs

Submit `submission_ensemble.zip`, `submission_heldout.zip`, and `submission_refit.zip` if budget allows, but they are all variants of **one methodology** (Partial/Layer-wise FT).

- `section_ablation_dev.json`, `dev_errors.json` (report analyses)
- `teacher_logits_cjpe.npz` (KD teacher signal)
- extra variant `submission_refit_rawthr.zip` (refit with the dev-tuned threshold)

- `top_snapshots.json` and `partialft_snapshot_*.pt`
- `teacher_logits_cjpe_refit_train_dev.npz` (future KD refit teacher)
- `submission_snapshot_ensemble.zip`
- `submission_snapshot_refit_ensemble.zip`
